# ToyAIKit

In [1]:
!uv add toyaikit

Resolved 144 packages in 797ms                                       
⠙ Preparing packages... (0/4)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/4)-------------------     0 B/646.61 KiB          
⠙ Preparing packages... (0/4)------------------- 16.00 KiB/646.61 KiB        
⠙ Preparing packages... (0/4)------------------- 32.00 KiB/646.61 KiB        
⠙ Preparing packages... (0/4)------------------- 48.00 KiB/646.61 KiB        
⠙ Preparing packages... (0/4)------------------- 61.70 KiB/646.61 KiB        
⠙ Preparing packages... (0/4)------------------- 77.70 KiB/646.61 KiB        
⠙ Preparing packages... (0/4)------------------- 93.70 KiB/646.61 KiB        
⠙ Preparing packages... (0/4)------------------- 93.70 KiB/646.61 KiB        
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/4)------------------- 93.70 KiB/646.61 KiB      

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)


In [3]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}


In [4]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": ["filename", "title", "description", "content"]
    }
}


In [7]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""


In [ ]:
from toyaikit.chat.runners import Tools

agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'title': {'type': 'string',
     'description': 'The title of the documentation entry'},
    'description': {'type': 'string',
     'description': 'A short description summarizing the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'title', 'description', 'content']}}]

In [11]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=llm_client
)


In [12]:
result = agent.loop('how can I create evidently dahsbord?')

In [13]:
print(result.last_message)

Based on the information gathered from the searches, here's a comprehensive guide on creating a dashboard:

### Steps to Create a Dashboard

1. **Setup Project**:
   - Ensure that you have connected to the platform and created a project. You must also have reports with evaluation results added to your project to populate the dashboard.

2. **Enter Edit Mode**:
   - Go to the Dashboard interface and enter "Edit" mode by clicking the relevant button in the top right corner.

3. **Add Tabs**:
   - Click the plus sign on the left to add a new Tab. You can create a custom Tab by selecting "empty" and entering a name.
   - Use pre-built Tabs if available, which contain preset panel combinations tailored to specific metrics.

4. **Add Panels**:
   - Click the "Add Panel" button. You can create various types of panels to visualize data:
     - **Text Panels**: For titles or descriptions.
     - **Counter Panels**: Display single values, ideal for quick insights.
     - **Pie Charts**: Visualiz

In [14]:
result.cost.total_cost

Decimal('0.0030819')

In [15]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received
